In [1]:
"""
Enumerate size-n MSTD sets (|A+A| > |A-A|) in Z/pZ up to affine equivalence.

Affine maps x -> a*x + b (a a unit mod p, b arbitrary) preserve both |A+A| and
|A-A|, so one representative is returned per class.  Since the affine group is
sharply 2-transitive, every set of >= 2 elements is equivalent to one containing
{0, 1}, so only the C(p-2, n-2) subsets through 0 and 1 need to be scanned..
"""
from itertools import combinations
from typing import List, Set


def _is_mstd(S, p) -> bool:
    """This function reports whether a set is MSTD in Z/pZ, i.e. whether its
    sumset is strictly larger than its difference set."""
    ss, dd = set(), set()
    for x in S:
        for y in S:
            ss.add((x + y) % p)
            dd.add((x - y) % p)
    return len(ss) > len(dd)


def _affine_canonical(S, p) -> Set[int]:
    """This function returns the canonical representative of a set under all affine
    maps x -> a*x + b, taking the lex-smallest sorted image so that affinely
    equivalent sets share one form."""
    Sl = list(S)
    best = None
    for a in range(1, p):                 # a ranges over the units (p prime)
        for b in range(p):
            img = tuple(sorted((a * x + b) % p for x in Sl))
            if best is None or img < best:
                best = img
    return set(best)


def mstd_sets_mod_p(p: int, n: int, verbose: bool = False) -> List[Set[int]]:
    """This function returns one representative per affine-equivalence class of
    size-n MSTD set in Z/pZ, scanning only the subsets that contain {0, 1} and
    deduplicating survivors by their affine-canonical form."""
    if p < 2:
        raise ValueError("p must be a prime >= 2")
    if n < 1 or n > p:
        return []
    if n < 2:
        # {0} is the only class and is never MSTD.
        return []

    seen = set()                          # canonical forms already recorded
    reps = []
    # Fix 0 and 1 in the set; choose the remaining n-2 elements from {2,...,p-1}.
    for combo in combinations(range(2, p), n - 2):
        S = (0, 1) + combo
        if _is_mstd(S, p):
            can = _affine_canonical(S, p)
            can_frozen = frozenset(can)   # frozenset is hashable for dedup
            if can_frozen not in seen:
                seen.add(can_frozen)
                reps.append(can)

    # Order the classes consistently by their sorted-tuple form.
    reps.sort(key=lambda s: tuple(sorted(s)))

    if verbose:
        print(f"Z/{p}Z, size {n}: {len(reps)} MSTD set(s) up to affine (ax+b)")
        for r in reps:
            print(f"  {{{', '.join(map(str, sorted(r)))}}}")
    return reps


KeyboardInterrupt: 

In [3]:
#Searches for all MSTD sets of size 8 in Z/37Z as of right now. Since only 1 appears, it must be affinely equivalent to Conway set.
mstd_sets_mod_p(37, 8, verbose=False)

[{0, 1, 2, 4, 27, 29, 30, 34}]